In [1]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification 
from peft import PeftConfig, LoraConfig, get_peft_model, TaskType, PeftModel, PeftModelForSequenceClassification

from captum.attr import IntegratedGradients, visualization as viz
import numpy as np

In [2]:
class MambaCaptumWrapper(nn.Module):
    """
    A CORRECTED wrapper for a Hugging Face model that works with embeddings.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, inputs_embeds):
        """
        This forward method now accepts pre-computed embeddings and passes
        them to the model.

        Args:
            inputs_embeds (torch.Tensor): Tensor of shape (batch_size, seq_len, hidden_size).

        Returns:
            torch.Tensor: A tensor of logits of shape (batch_size, num_labels).
        """
        # We pass `inputs_embeds` directly. The model will use these instead of
        # running its own embedding layer. We set input_ids to None.
        outputs = self.model(input_ids=None, inputs_embeds=inputs_embeds)
        return outputs.logits


def load_base_model(model_name: str, task_type: str = "classification") -> AutoModelForSequenceClassification:
    """
    Loads the base model and wraps its forward method to gracefully handle
    the 'attention_mask' argument passed by the PEFT wrapper.
    """
    if task_type == "classification":
        print(f"Loading base model for classification from {model_name}")
        base_model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            trust_remote_code=True,
            num_labels=2,
            id2label={0: "NEGATIVE", 1: "POSITIVE"},
            label2id={"NEGATIVE": 0, "POSITIVE": 1},
        )
    else:  # regression
        # This part is for completeness, assuming you might have a logger
        print(f"Loading base model for regression from {model_name}")
        base_model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            trust_remote_code=True,
            num_labels=1,
            problem_type="regression",
        )

    # Keep a reference to the original forward method
    original_forward = base_model.forward

    # Define a new 'forgiving' forward method
    def forgiving_forward(*args, **kwargs):
        # The PEFT wrapper will pass 'attention_mask', but Caduceus doesn't
        # use it. We simply remove it from the arguments before calling
        # the model's original forward pass.
        kwargs.pop('attention_mask', None)
        kwargs.pop('output_attentions', None)
        kwargs.pop('output_hidden_states', None)
        return original_forward(*args, **kwargs)

    # Override the base model's forward method with our new one
    base_model.forward = forgiving_forward

    return base_model

def create_peft_model(base_model: AutoModelForSequenceClassification) -> PeftModelForSequenceClassification:
    peft_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        inference_mode=False,
        r=8,
        lora_alpha=32,
        lora_dropout=0.1,
        target_modules=["x_proj", "in_proj", "out_proj"],
    )
    return get_peft_model(base_model, peft_config)

In [3]:
PEFT_MODEL_PATH = "../../../results/PlantCAD2_tasks/exp-leaf-bin/pcv2-l24-d0768-checkpoints-lr-1e-4/checkpoint-7975/" 
model_name = 'kuleshov-group/compo-cad2-l24-dna-chtk-c8192-v2-b2-NpnkD-ba240000'
base_model = load_base_model(model_name)
model = PeftModel.from_pretrained(base_model, PEFT_MODEL_PATH)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)

Loading base model for classification from kuleshov-group/compo-cad2-l24-dna-chtk-c8192-v2-b2-NpnkD-ba240000


/home/jz963/miniconda3/envs/transformers/lib/python3.11/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of the model checkpoint at kuleshov-group/compo-cad2-l24-dna-chtk-c8192-v2-b2-NpnkD-ba240000 were not used when initializing CaduceusForSequenceClassification: ['lm_head.complement_map', 'lm_head.lm_head.weight']
- This IS expected if you are initializing CaduceusForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing CaduceusForSequenceClassification from the checkpoint

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): CaduceusForSequenceClassification(
      (caduceus): Caduceus(
        (backbone): CaduceusMixerModel(
          (embeddings): CaduceusEmbeddings(
            (word_embeddings): RCPSEmbedding(
              (embedding): Embedding(8, 768)
            )
          )
          (layers): ModuleList(
            (0-23): 24 x RCPSMambaBlock(
              (mixer): RCPSWrapper(
                (submodule): BiMambaWrapper(
                  (mamba_fwd): Mamba2(
                    (in_proj): lora.Linear(
                      (base_layer): Linear(in_features=768, out_features=3224, bias=False)
                      (lora_dropout): ModuleDict(
                        (default): Dropout(p=0.1, inplace=False)
                      )
                      (lora_A): ModuleDict(
                        (default): Linear(in_features=768, out_features=8, bias=False)
                      )
                      (lora_B): Module

In [4]:
embedding_layer = model.base_model.caduceus.backbone.embeddings.word_embeddings.embedding

In [5]:
wrapped_model = MambaCaptumWrapper(model)
ig = IntegratedGradients(wrapped_model)

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [7]:
try:
    # The full module that does the RCPS logic
    word_embedding_module = model.base_model.caduceus.backbone.embeddings.word_embeddings
    
    # The innermost nn.Embedding layer
    embedding_layer = word_embedding_module.embedding
    
    # The layer/buffer that handles the reverse-complement mapping of token IDs
    rc_layer = word_embedding_module.rc
    
    print("Successfully found all embedding components.")
except AttributeError:
    print("Could not find embedding components. Please check model structure.")
    exit()

# (Wrapper and IG initialization are the same)
wrapped_model = MambaCaptumWrapper(model)
ig = IntegratedGradients(wrapped_model)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Successfully found all embedding components.


In [8]:
# --- Prepare inputs (DNA sequence, input_ids, baseline_ids are the same) ---
dna_sequence = 'CAATTTCCCTTTCACATACTATTGAGGATTTGTGAGGTTGCCAGTTTTACTTGATCCTCCCATGCTCAAACTTCATATCCCATCTCTATTTAGGTCCAGAAACCAAGTGCCGAAATCCTAAATCATTTTGGGTTTTCTGGTCTGCACAACACTACTGCGTACATATGCGGCACTACTGCGCCCTAGCACTGTCGGATTTTTCTCGGCATCGCTAGACCTTAAAATATCTGTTGGGACACCAGGGCCTCGAATTGTTTATTCTCTATGTCCCAACAGTCACCTTATGCGTATCCTGACAGGAGAATATGGTAGAAAGACTACTTATGTCCTCTTTGCCTCTCTTGGAGCCCCATCCCTTCACTCACATTCCATCCCCAAGCTCAACATATTGCTCTCCCCATTTCAAACTGAGAAGAACAACAAGAACACTAATTTCTTTCTCAGTTCTTCATTCTCTTGAGCATTGAAGTTCTCTAGGCCGACATTCTTCACTCAATCCTTAGAGCTTGCTCCTAGTCGGCAATTGCATTCTCCACGAGCTGCAAGACTTTGTCGCAGACTTAGATTGCTGTAAGTTATCCTTTGAATTAGTGAAAAACCTTCTCATCTCAAGAGCGAGAGCTCTTGACTTGAGATCGAGAATGGAATGAGACCCTTTGCCTTAGTGGTTTTTCTCACAACATGAAGGTAGACAAGCCTTTGTGACGAGTTAAACCACTGGATAAATATTGTGTCTCTGTGTGCTTGATCTAAAATGAACACGTGATTAGCTAAGGTTTAAGTGACATCTAATTTGCCATCTGTTGCTTCGACTTGCTAGTGCCCCTTCCCAGCTCTTAAATCAATTCATCTTGATGGGCTGGACTGATCGGGGCCGACAAACAGTTAAAGTGGCGGCAACGGGCCGGCCTCCTATATCCTTCCTTATATGGGCTAGGCTGGATTGGGCTGGCCCTGAAATGGAAAGGAAGTTCGTCCTCTCTCTCACGGTCTCACCTCTCACGTCCAGAGAGAGAGAGAGAGGGTTATCAATCAATCTATCTACATGTATGCCTCTGGTCTTTAGAGGAACCAGCACGTGAGCTTGGCAAGAGTTACTGATCTTTTGTAAATTTTGATGATAGGAGCCGAGCCGTCACAGTGTGTTGTTGCTGTTACAGTTGAGTTTGTTGGCATGTCATGAGATTGTCTGGTACAGTTTCAACAACTGATGAATGAACATGGTCACTGGAAGATGCTGCTGACGGAGCAGTCGTCGTGCACACGCCAGATGGTCTCACCGAGGAGGTTAGCGACGGAGAGCACGGTGAGCTGTGCACCGGCACGGTGTTGGTGATGATCACCTCCTGGAAGAGGTCGCTGGAGAGCCTCTGGAGCGCGGGTGGGCTGTTGAGGACCGCGTGGGTGCTGCAGGTGTACACAACTCGCGCTCCCTTTGTGCAGCAGCTCGGCGCCCTTGGAGATGTTCTCGGTCGTGTTGATCATGCCGTCCACCATGACGACGACCTTTCCCCTGACATCATGGATGAGGTGCACCACCTCGGCCTGGTTGTGGCCTGGTCGCCTCTTGTCGACGATCGCCAACGGCGCGTCGGAGAGCATGACGACGACCTTTCCCCTGACATCATCGATGAGGTGCACCACCTCGGCCTGGTTGTGGCCCGGTCGCCTCTTGTCGACGATCGCCAGCGGCGCGTCAGAGAGCTTCTTGGCGAAGGCGCGCGGCCTGGCCACCCCTCCCATGTCCGGCGACACCACCACCACGTCCTCGGGGCAGATGGTCTTGCTGGCTAGGTAGTCGAGGATGATGGGCTGGCTGTGGACACGTGGTCCACGGGGATGTCGAAGTAGTCGATGGACTGACCCGAGTGGAGTTCGCAGGCCAGCACGCGGTGGGCGCCGACCTCCATGATGAGGTTGGCCACCAGCTTGGCGGCGATGGAGCCGCGCCCCCTGCACCTTCTTATCGGCCCTGGCGTAGCCGAAGTAGGGGATGCCAGCGTTGATGTTCTTGGCGGAGGCCCTTCGGCAGGCATCGATCATGATGA'
inputs = tokenizer(dna_sequence, return_tensors="pt")
input_ids = inputs['input_ids'].to(device)
pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
baseline_ids = torch.full_like(input_ids, pad_token_id)


# --- Manually replicate the FULL RCPS embedding logic ---
print("Manually creating RCPS embeddings...")

# 1. Process the main input_ids
fwd_out = embedding_layer(input_ids)
rc_ids = rc_layer(input_ids)
rc_out = torch.flip(embedding_layer(rc_ids), dims=[-2, -1])
inputs_embeds = torch.cat([fwd_out, rc_out], dim=-1)

# 2. Process the baseline_ids with the same logic
baseline_fwd_out = embedding_layer(baseline_ids)
baseline_rc_ids = rc_layer(baseline_ids)
baseline_rc_out = torch.flip(embedding_layer(baseline_rc_ids), dims=[-2, -1])
baseline_embeds = torch.cat([baseline_fwd_out, baseline_rc_out], dim=-1)

print(f"Shape of final inputs_embeds: {inputs_embeds.shape}") # Should have 2 * d_model as the last dim

# --- Calculate attributions (this should now work) ---
target_class_index = 1
print("Calculating attributions...")
attributions = ig.attribute(inputs_embeds, baselines=baseline_embeds, target=target_class_index, internal_batch_size=2)

Manually creating RCPS embeddings...
Shape of final inputs_embeds: torch.Size([1, 2048, 1536])
Calculating attributions...


In [16]:
with torch.no_grad():
    outputs = model(input_ids)

probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)

predicted_class_index = torch.argmax(probabilities, dim=-1).item()
predicted_class_prob = probabilities[0, predicted_class_index].item()
predicted_class_name = model.config.id2label[predicted_class_index]


attributions_summary = attributions.sum(dim=-1).squeeze(0)
attributions_summary = attributions_summary / torch.norm(attributions_summary)
attributions_summary = attributions_summary.cpu().detach().numpy()

tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze(0).tolist())


# --- Create the VisualizationDataRecord with the CORRECT values ---
visual_record = viz.VisualizationDataRecord(
    word_attributions=attributions_summary,
    pred_prob=predicted_class_prob,         # Use the calculated probability
    pred_class=predicted_class_name,        # Use the calculated class name
    true_class="?",                         # You can put the real label here if you know it
    attr_class=model.config.id2label[target_class_index], # The class we attributed to
    attr_score=attributions_summary.sum(),
    raw_input_ids=tokens,
    convergence_score=None # Captum can calculate this, but it's optional
)


# --- Visualize the final result ---
print("Visualizing attribution scores...")
viz.visualize_text([visual_record])

Visualizing attribution scores...


In [10]:
attributions_summary = attributions.sum(dim=-1).squeeze(0)
attributions_summary = attributions_summary / torch.norm(attributions_summary)
attributions_summary = attributions_summary.cpu().detach().numpy()

In [11]:
attributions_summary.shape

(2048,)

In [12]:
tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze(0).tolist())

In [14]:
visual_record = viz.VisualizationDataRecord(
    word_attributions=attributions_summary,
    pred_prob=None,
    pred_class="POSITIVE",
    true_class=None,
    attr_class="POSITIVE",
    attr_score=attributions_summary.sum(),
    raw_input_ids=tokens,
    convergence_score=None
)

In [15]:
viz.visualize_text([visual_record])

TypeError: unsupported format string passed to NoneType.__format__